In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [6]:
# Dataset and DataLoader
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# image => scale (0,1) ==> normalize ==> (-1,1)
transform = transforms.Compose([
    transforms.ToTensor(), # images --> tensor + scale
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = CIFAR10(root="./data", train=True, download=False, transform=transform)
testset = CIFAR10(root="./data/", train=False, download=False, transform=transform)

In [7]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [8]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

# Build the CNN

In [10]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernel size = 2, stride = 2

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flattening
        x = self.fc_layers(x)
        
        return x

In [11]:
model = CNN()

In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Training CNN

In [18]:
epochs = 10
for epoch in range(epochs):
    model.train()
    epoch_train_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()
        
        output = model.forward(images) # FP
        loss = criterion(output, labels) # loss fnx
        loss.backward() # BP
        optimizer.step() # update params

        epoch_train_loss += loss.item()

    print(f"epoch={epoch+1} & loss={epoch_train_loss/len(trainloader)}")

epoch=1 & loss=0.11149565707129971
epoch=2 & loss=0.10439448869463218
epoch=3 & loss=0.09017975927899828
epoch=4 & loss=0.09202409711192407
epoch=5 & loss=0.08447511450481622
epoch=6 & loss=0.07853190285865874
epoch=7 & loss=0.07286507977664594
epoch=8 & loss=0.06701424330829636
epoch=9 & loss=0.0715647108664991
epoch=10 & loss=0.06042177683841723


In [19]:
# Evaluate our CNN

correct_labels = 0
total_labels = 0

model.eval()
with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuraccy = {(correct_labels/total_labels)*100}")

accuraccy = 74.65
